# 점자블록 파손 탐지 - YOLOv8s 최종 학습
**텐서프로그래밍 기말 프로젝트 | 하나됨 조**

| 항목 | 내용 |
|------|------|
| 모델 | YOLOv8s (ultralytics) |
| 데이터셋 | Roboflow v2 (4,144장, 2 classes) |
| 클래스 | 0: normal (정상) / 1: damaged (교체폐기) |
| 학습 환경 | Google Colab (NVIDIA A100-SXM4-40GB) |
| 최종 성능 | mAP50: 0.848 / mAP50-95: 0.684 / Recall: 0.776 |

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 패키지 설치

In [ ]:
!pip install ultralytics roboflow -q

## 2. 데이터셋 다운로드 (Roboflow v2)

- 총 4,144장 (Train 3,049 / Valid 962 / Test 133)
- 라벨 기준 통일 완료 (손상 스팟 → 블록 전체)
- 오기입 클래스(0) 제거, Auto-Orient + Resize 640×640 적용

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="MvcagQ61vrcJFQxn3SkI")
project = rf.workspace("junseo").project("braille-sidewalk-blocks-r2jn2-uvvvj")
dataset = project.version(2).download("yolov8")

## 3. 모델 학습

| 파라미터 | 값 | 설명 |
|---------|-----|------|
| model | YOLOv8s | nano → small 업그레이드 (11M params) |
| epochs | 100 | patience=20 조기종료 포함 |
| imgsz | 640 | 입력 해상도 |
| batch | 16 | A100 기준 |
| mixup | 0.2 | 두 이미지 혼합 증강 |
| degrees | 15 | 회전 ±15° 증강 |
| hsv_s/v | 0.7/0.4 | 채도·명도 변화 (조명 다양성) |

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    project="tactile_block",
    name="yolov8s_v2",
    mosaic=1.0,
    mixup=0.2,
    degrees=15,
    translate=0.2,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

## 4. 성능 평가 (Test Set)

In [ ]:
best_model = YOLO("/content/runs/detect/tactile_block/yolov8s_v2/weights/best.pt")

test_results = best_model.val(
    data=f"{dataset.location}/data.yaml",
    split="test"
)

print(f"Precision : {test_results.box.mp:.4f}")
print(f"Recall    : {test_results.box.mr:.4f}")
print(f"mAP50     : {test_results.box.map50:.4f}")
print(f"mAP50-95  : {test_results.box.map:.4f}")

## 5. 학습 결과 시각화

In [ ]:
from IPython.display import Image, display

print("=== 학습 곡선 ===")
display(Image("/content/runs/detect/tactile_block/yolov8s_v2/results.png"))

print("=== Confusion Matrix ===")
display(Image("/content/runs/detect/tactile_block/yolov8s_v2/confusion_matrix.png"))

## 6. 모델 저장 (Google Drive)

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

shutil.copy(
    "/content/runs/detect/tactile_block/yolov8s_v2/weights/best.pt",
    "/content/drive/MyDrive/braille_best_s_v2.pt"
)
print("모델 저장 완료: Google Drive/braille_best_s_v2.pt")